# Chapter 6 - Statistical Machine Learning

## Chapter summary
Statistical machine learning methods are **data-driven, non-parametric** prediction algorithms that learn from the data rather than from a fixed equation. Topics:

1. **K-Nearest Neighbors (KNN)** - classify a record by a vote of the $K$ most similar records.
2. **Tree models** (CART) - recursive partitioning into homogeneous regions; impurity measures (Gini, entropy), pruning.
3. **Bagging and the random forest** - average many trees built on bootstrap samples (plus random predictor subsets).
4. **Boosting** (gradient boosting, XGBoost) - build trees sequentially, each fixing the previous errors; regularisation to prevent overfitting.
5. **Hyperparameters and cross-validation** - tuning model settings without overfitting.

We use Lending Club loan data to predict `default` vs `paid off`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"

def get_data(name):
    """Download a dataset of the book (once) into ./data and return its path."""
    path = Path("data") / name
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + name, path)
    return path

sns.set_style("whitegrid")

loan200 = pd.read_csv(get_data("loan200.csv"))
loan3000 = pd.read_csv(get_data("loan3000.csv"))
loan_data = pd.read_csv(get_data("loan_data.csv.gz"))
loan_data = loan_data.drop(columns=[c for c in loan_data.columns if c.startswith("Unnamed")])

## Background: the bias-variance trade-off

Every method in this chapter is a way of balancing two sources of prediction error:
* **Bias** - error from a model that is too simple to capture the real pattern (a straight line through a curve; a tree of depth 1; KNN with a huge $K$). The model *underfits*: it is wrong in the same way no matter which sample it sees.
* **Variance** - error from a model that is so flexible that it follows the noise of the particular training sample (a very deep tree; KNN with $K=1$). The model *overfits*: refit it on a different sample and its predictions change a lot.

Expected test error $\approx \text{bias}^2 + \text{variance} + \text{irreducible noise}$. Making a model more flexible lowers bias but raises variance, so the best model sits in the middle, and we find it with **cross-validation** (Section 5). The methods below use this trade-off in different ways: a single **tree** has low bias but high variance; **bagging / random forests** keep the low bias and *average away* the variance; **boosting** starts with high-bias weak learners and *reduces the bias* step by step, using regularisation to keep variance in check. All of them are **non-parametric** - the shape of the model is learned from the data rather than being a fixed equation with a few coefficients, as in regression.

## 1. K-Nearest Neighbors

**Theory.** To classify a new record: (1) find the $K$ records most similar (closest in feature space), (2) take the majority class (or class proportions as a probability).
* **Distance metric**: usually **Euclidean** $\sqrt{\sum (x_j-u_j)^2}$ or Manhattan $\sum |x_j-u_j|$.
* **Standardisation (z-scores)** is needed so that variables with large scales don't dominate the distance.
* **Choice of $K$**: small $K$ -> flexible, overfits noise; large $K$ -> smooth, underfits. Pick by cross-validation.
* KNN can also be used as a **feature engine**: its predicted probability becomes a feature for another model.

Example: predict whether a new loan (payment_inc_ratio=9, dti=22.5) will default using 20 examples.

In [ ]:
predictors = ["payment_inc_ratio", "dti"]
outcome = "outcome"
newloan = loan200.loc[0:0, predictors]
X = loan200.loc[1:, predictors]; y = loan200.loc[1:, outcome]

knn = KNeighborsClassifier(n_neighbors=20).fit(X, y)
print("Prediction:", knn.predict(newloan)[0])
print("Probabilities:", dict(zip(knn.classes_, knn.predict_proba(newloan)[0])))

nbrs = knn.kneighbors(newloan)
print("Neighbours:\n", X.iloc[nbrs[1][0], :])

In [ ]:
# Why standardisation matters: the 5 nearest neighbours of the first loan, with and without scaling
loan_data["outcome"] = pd.Categorical(loan_data["outcome"], categories=["paid off", "default"])
predictors = ["loan_amnt", "annual_inc", "revol_bal", "open_acc", "dti", "revol_util"]
X = loan_data[predictors].fillna(0); y = loan_data["outcome"]
new_rec = X.iloc[[0]]
X_rest, y_rest = X.iloc[1:], y.iloc[1:]
print("Loan to classify:\n", new_rec.round(1).to_string(index=False))

knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_rest, y_rest)
print("\nNeighbours WITHOUT scaling (annual_inc and revol_bal, measured in thousands, dominate the distance):\n",
      X_rest.iloc[knn_raw.kneighbors(new_rec)[1][0]].round(1))

sc = StandardScaler().fit(X)
knn_z = KNeighborsClassifier(n_neighbors=5).fit(sc.transform(X_rest), y_rest)
print("\nNeighbours WITH standardisation (every variable counts equally):\n",
      X_rest.iloc[knn_z.kneighbors(sc.transform(new_rec))[1][0]].round(1))

In [ ]:
# Choosing K by cross-validation on a 3000-loan sample
Xk = StandardScaler().fit_transform(loan3000[["borrower_score", "payment_inc_ratio"]])
yk = loan3000["outcome"]
ks = range(1, 60, 4)
scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k), Xk, yk, cv=5).mean() for k in ks]
plt.figure(figsize=(5, 3.5)); plt.plot(list(ks), scores, "o-")
plt.xlabel("K"); plt.ylabel("CV accuracy"); plt.title("Choosing K"); plt.show()

**Limits of KNN and KNN as a feature engine.** KNN has no training step (it stores the data), so *prediction* is slow on large data, and it suffers from the **curse of dimensionality**: as the number of predictors grows, all points become almost equally far apart, so "nearest" loses its meaning - use few, relevant predictors. Because it is cheap to set up, KNN is often used as a **feature engine**: its (out-of-fold) predicted probability is added as a new column to another model. The cell below builds such a feature honestly, with cross-validated predictions, and checks how well it separates defaults on its own.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

classes = list(np.sort(yk.unique()))
knn_feature = cross_val_predict(KNeighborsClassifier(n_neighbors=20), Xk, yk, cv=5,
                                method="predict_proba")[:, classes.index("default")]
print(f"AUC of the KNN feature alone: {roc_auc_score(yk == 'default', knn_feature):.3f}")

## 2. Tree Models

**Theory.** A **decision tree** repeatedly splits the data on a predictor/value so that each resulting **partition** is as *pure* (homogeneous in the outcome) as possible; the path is a set of if-then rules, and a leaf predicts the majority class (or mean for regression).

* **Recursive partitioning**: for every predictor and split value, measure the reduction in impurity; choose the best split; repeat inside each partition.
* **Impurity measures**: **Gini** $I=p(1-p)$ (for 2 classes; generally $1-\sum p_k^2$) and **entropy** $-\sum p_k\log_2 p_k$. Split to maximise the impurity decrease.
* **Overfitting & stopping**: a fully grown tree memorises the training data. Control it with `max_depth`, `min_samples_leaf`, `min_impurity_decrease` or by **pruning** (cost-complexity, `ccp_alpha`) chosen by cross-validation.
* Pros: interpretable, handle non-linearity/interactions, no scaling needed. Cons: unstable (high variance), piecewise-constant predictions.

In [ ]:
predictors = ["borrower_score", "payment_inc_ratio"]
loan_tree = DecisionTreeClassifier(criterion="entropy", min_impurity_decrease=0.003)
loan_tree.fit(loan3000[predictors], loan3000["outcome"])

plt.figure(figsize=(14, 7))
plot_tree(loan_tree, feature_names=predictors, class_names=loan_tree.classes_, filled=True, impurity=False, fontsize=8)
plt.show()

**Reading the tree as rules, and how a split is chosen.** The fitted tree is just a set of if-then rules, which is why trees are popular for explaining decisions to non-technical people. To choose a split the algorithm compares the **impurity** of the parent node with the size-weighted impurity of the two children; the difference is the **information gain**. For a node with class proportions $p_k$: Gini $=1-\sum p_k^2$ (0 = pure) and entropy $=-\sum p_k\log_2 p_k$ (0 = pure, 1 = 50/50 for two classes). We recompute the gain of the root split by hand:

In [ ]:
from sklearn.tree import export_text
print(export_text(loan_tree, feature_names=predictors, max_depth=3))

def gini(s):
    p = s.value_counts(normalize=True); return 1 - (p ** 2).sum()

def entropy(s):
    p = s.value_counts(normalize=True); return -(p * np.log2(p)).sum()

y3 = loan3000["outcome"]
feature, threshold = predictors[loan_tree.tree_.feature[0]], loan_tree.tree_.threshold[0]
left, right = y3[loan3000[feature] <= threshold], y3[loan3000[feature] > threshold]
print(f"Root split: {feature} <= {threshold:.3f}  ({len(left)} records left, {len(right)} right)")
for name, f in (("Gini", gini), ("Entropy", entropy)):
    children = (len(left) * f(left) + len(right) * f(right)) / len(y3)
    print(f"{name:8s} parent={f(y3):.3f}  weighted children={children:.3f}  gain={f(y3) - children:.3f}")

**Overfitting in trees (high variance).** The deeper the tree, the better it fits the training data - but past a point test accuracy stops improving or falls, because the tree is memorising noise:

In [ ]:
from sklearn.model_selection import train_test_split
Xtr3, Xte3, ytr3, yte3 = train_test_split(loan3000[predictors], loan3000["outcome"], test_size=0.3, random_state=1)
for depth in (1, 2, 4, 8, 16, None):
    t = DecisionTreeClassifier(max_depth=depth, random_state=1).fit(Xtr3, ytr3)
    print(f"max_depth={str(depth):>4}  train acc={accuracy_score(ytr3, t.predict(Xtr3)):.3f}  test acc={accuracy_score(yte3, t.predict(Xte3)):.3f}")

In [ ]:
# Cost-complexity pruning chosen by CV
path = DecisionTreeClassifier().cost_complexity_pruning_path(loan3000[predictors], loan3000["outcome"])
alphas = path.ccp_alphas[:-1][::max(1, len(path.ccp_alphas) // 25)]
cv = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=1), loan3000[predictors],
                      loan3000["outcome"], cv=5).mean() for a in alphas]
best_alpha = alphas[int(np.argmax(cv))]
print("Best ccp_alpha:", best_alpha, " CV accuracy:", round(max(cv), 3))

## 3. Bagging and the Random Forest

**Theory.**
* **Ensemble**: combine many models; the average is usually better than a single one.
* **Bagging** (bootstrap aggregating): fit a model on each of $B$ bootstrap resamples and average (regression) or vote (classification). It reduces **variance**.
* **Random forest**: bagging with decision trees *plus* a random subset of predictors considered at each split, which decorrelates the trees.
* **Out-of-bag (OOB) error**: each tree leaves out ~1/3 of the records; predicting them gives a free validation estimate.
* **Variable importance**: how much accuracy/impurity decreases when a predictor is permuted/used. Hyperparameters: `n_estimators`, `max_features`, `min_samples_leaf`.

**Bagging by hand.** Below we build the ensemble ourselves: 100 unpruned trees, each trained on a bootstrap resample of the training data (the bootstrap from Chapter 2), then a majority vote. Compare it with one unpruned tree on the same test data - the single tree has high variance; averaging many of them cancels much of it.

In [ ]:
from sklearn.utils import resample

single = DecisionTreeClassifier(random_state=1).fit(Xtr3, ytr3)
votes = []
for b in range(100):
    Xb, yb = resample(Xtr3, ytr3, random_state=b)
    votes.append(DecisionTreeClassifier(random_state=1).fit(Xb, yb).predict(Xte3) == "default")
bagged = np.where(np.mean(votes, axis=0) > 0.5, "default", "paid off")
print(f"Single unpruned tree test accuracy : {accuracy_score(yte3, single.predict(Xte3)):.3f}")
print(f"Bagged ensemble (100 trees)        : {accuracy_score(yte3, bagged):.3f}")

In [ ]:
predictors = ["loan_amnt", "term", "annual_inc", "dti", "payment_inc_ratio", "revol_bal", "revol_util",
              "purpose", "delinq_2yrs_zero", "pub_rec_zero", "open_acc", "grade", "emp_length",
              "purpose_", "home_", "emp_len_", "borrower_score"]
loan_X = pd.get_dummies(loan_data[[c for c in predictors if c in loan_data.columns]], drop_first=True).astype(float)
loan_X = loan_X.fillna(loan_X.median())
loan_y = (loan_data["outcome"] == "default").astype(int)
X_train, X_test, y_train, y_test = train_test_split(loan_X, loan_y, test_size=0.3, random_state=1, stratify=loan_y)

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, oob_score=True, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"OOB accuracy : {rf.oob_score_:.3f}")
print(f"Test accuracy: {accuracy_score(y_test, rf.predict(X_test)):.3f}  AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]):.3f}")

In [ ]:
imp = pd.Series(rf.feature_importances_, index=loan_X.columns).sort_values().tail(12)
imp.plot.barh(figsize=(6, 4)); plt.title("Random forest: variable importance (Gini)"); plt.tight_layout(); plt.show()

**Two kinds of variable importance.** The Gini importance above (mean decrease in impurity) is computed on the *training* data and tends to favour variables with many distinct values. **Permutation importance** is more trustworthy: shuffle one predictor in the *test* data and measure how much the model's AUC drops - a big drop means the model really relied on that variable.

In [ ]:
from sklearn.inspection import permutation_importance

idx = X_test.sample(3000, random_state=1).index
pi = permutation_importance(rf, X_test.loc[idx], y_test.loc[idx], n_repeats=3, scoring="roc_auc",
                            random_state=1, n_jobs=-1)
pd.Series(pi.importances_mean, index=loan_X.columns).sort_values().tail(10).plot.barh(figsize=(6, 4))
plt.title("Permutation importance (drop in AUC)"); plt.tight_layout(); plt.show()

## 4. Boosting

**Theory.** Boosting builds a sequence of **weak learners** (small trees), each concentrating on the records the previous ones got wrong.
* **AdaBoost** re-weights misclassified records.
* **Gradient boosting** fits each new tree to the **residuals / negative gradient** of the loss, adding it with a **learning rate** $\eta$: $F_m = F_{m-1}+\eta\, h_m$.
* **XGBoost** (extreme gradient boosting) adds regularisation: $L1/L2$ penalties on leaf weights (`reg_alpha`, `reg_lambda`), row/column subsampling (`subsample`, `colsample_bytree`), `max_depth`, and early stopping.
* **Overfitting control** is crucial: boosting can drive training error to zero; monitor validation error. Compared with random forest, boosting is usually more accurate but needs more tuning.

**Gradient boosting from scratch (regression version).** Start with a constant prediction (the mean). Then repeat: compute the residuals (what the model still gets wrong), fit a *small* tree to those residuals, and add a fraction $\eta$ (the learning rate) of its predictions to the model. Each tiny tree is a weak learner, but together they trace out the curve - bias falls round after round.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(1)
x_s = np.sort(rng.uniform(0, 6, 200)).reshape(-1, 1)
y_s = np.sin(x_s[:, 0]) + rng.normal(0, 0.3, 200)

pred, eta = np.full_like(y_s, y_s.mean()), 0.3
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for m in range(1, 51):
    tree = DecisionTreeRegressor(max_depth=2).fit(x_s, y_s - pred)     # fit the current residuals
    pred = pred + eta * tree.predict(x_s)
    if m in (1, 5, 50):
        ax = axes[(1, 5, 50).index(m)]
        ax.scatter(x_s[:, 0], y_s, s=8, alpha=0.5); ax.plot(x_s[:, 0], pred, color="C3")
        ax.set_title(f"After {m} tree(s): MSE={np.mean((y_s - pred) ** 2):.3f}")
plt.tight_layout(); plt.show()

In [ ]:
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1, subsample=0.63, colsample_bytree=0.8,
                    reg_lambda=1.0, eval_metric="logloss", random_state=1)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(f"XGBoost test accuracy: {accuracy_score(y_test, xgb.predict(X_test)):.3f}  "
      f"AUC: {roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]):.3f}")

hist = xgb.evals_result()["validation_0"]["logloss"]
plt.figure(figsize=(5, 3.5)); plt.plot(hist)
plt.xlabel("Boosting round"); plt.ylabel("Validation log-loss"); plt.title("XGBoost learning curve"); plt.show()

**Overfitting demo:** a deep, strongly-fit model with a large learning rate reaches a lower training error but higher test error than a regularised one.

In [ ]:
for params in ({"max_depth": 3, "learning_rate": 0.1, "reg_lambda": 1},
               {"max_depth": 12, "learning_rate": 0.5, "reg_lambda": 0}):
    m = XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=1, **params).fit(X_train, y_train)
    print(params, f"| train acc={accuracy_score(y_train, m.predict(X_train)):.3f}  test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

## 5. Hyperparameters and Cross-Validation

**Theory.** **Hyperparameters** are settings chosen *before* training (tree depth, number of neighbours, learning rate). Choosing them by test-set performance leaks information, so use **k-fold cross-validation**: split the training data into $k$ folds, train on $k-1$, validate on the held-out fold, rotate and average. A **grid search** evaluates every combination; **random search** or Bayesian optimisation scale better. Finally, evaluate the chosen model once on a truly held-out **test set**.

In [ ]:
param_grid = {"max_depth": [2, 3, 5], "learning_rate": [0.05, 0.1, 0.3], "n_estimators": [100, 300]}
grid = GridSearchCV(XGBClassifier(eval_metric="logloss", subsample=0.63, random_state=1),
                    param_grid, scoring="roc_auc", cv=3, n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_, " CV AUC:", round(grid.best_score_, 3))
print("Held-out test AUC:", round(roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1]), 3))

## Key takeaways
* **KNN**: simple, similarity-based; standardise the predictors and tune $K$.
* **Trees**: interpretable rules from recursive partitioning; control depth/pruning to avoid overfitting.
* **Random forest** (bagging + random features) lowers variance and gives OOB error and variable importance.
* **Boosting / XGBoost** often gives the best accuracy; it needs regularisation and careful tuning.
* Tune hyperparameters with **cross-validation** and keep a test set untouched.